# 01 — Prime Leaves of Telperion

**The Zero Tree / Telperion / Un-Extinctable Bulk — Notebook 1 of 4**

---

The primes are the leaves at k=0 (ℝ, north pole).  
They cannot fall because they have no non-trivial factorization — Fermat's  
Nightmare finds no ZD pair to express them as.

Each prime p maps to a **N-shape**: `p mod 16` → sedenion basis element e_k.  
This is the prime's address in the tree.

**Three leaf classes:**
- **Monster gap** (p ≡ 1, 11, 15 mod 16): the Telperion silver leaves.  
  No Niemeier A/D/E root system can reach these N-shapes. Monster fills them.
- **Niemeier** (all other odd p mod 16): covered by Niemeier root systems.
- **Leech-adjacent / even** (p ≡ 0, 2, 4, ... mod 16): only p=2 (even prime).

**Fractal character:**  
The prime gaps are not uniform — they grow, cluster, and oscillate.  
Those oscillations are driven by Riemann zeros: the same zeros that form  
the lens of the Zero Tree (wiki #72, *The Cosmic Telescope*).
The boundary between 'surviving' and 'fallen' at each CD level  
inherits this fractal structure.

In [ ]:
import sys, os
sys.path.insert(0, os.path.join('..', 'engine'))
from telperion_engine import (
    prime_sieve, prime_nshape, classify_prime,
    prime_gap_fractal, NIEMEIER_GAP, PRIME_SECTOR, MOONSHINE_PRIMES,
)

N      = 1000
primes = prime_sieve(N)
print(f'Primes ≤ {N}: {len(primes)}')
print(f'First 20: {primes[:20]}')

## N-shape distribution (p mod 16)

In [ ]:
from collections import Counter

ns_counts = Counter(p % 16 for p in primes)
total     = len(primes)

print(f'{'N-shape':>8}  {'Count':>6}  {'Frac%':>7}  Class')
print('-' * 55)
for ns in range(16):
    cnt = ns_counts.get(ns, 0)
    frac = cnt / total * 100
    cls = classify_prime(next((p for p in primes if p % 16 == ns), 2))['classification']
    gap_mark = ' ← SILVER LEAF' if ns in NIEMEIER_GAP else ''
    print(f'  e{ns:2d}      {cnt:6d}  {frac:7.3f}%  {cls}{gap_mark}')

## Dirichlet asymptotic vs. actual density

By Dirichlet's theorem on primes in arithmetic progressions:  
`π(x; 16, ns) ~ li(x) / φ(16)` for `gcd(ns, 16) = 1`.  

`φ(16) = 8` (the 8 odd residues coprime to 16 are the prime-sector N-shapes).  
Asymptotic density per odd N-shape: `1/8 = 12.5%`.  
Even N-shapes: only p=2 qualifies → density → 0 at large N.

**Deviations from 12.5%** are the oscillations driven by L-function zeros.  
These are the fractal oscillations of the boundary.

In [ ]:
import matplotlib.pyplot as plt
import matplotlib.patches as mpatches
import numpy as np

odd_ns    = [ns for ns in range(16) if ns % 2 == 1]
dirichlet = 100.0 / 8   # = 12.5%

actual = [ns_counts.get(ns, 0) / total * 100 for ns in odd_ns]
colors = ['silver' if ns in NIEMEIER_GAP else '#5588cc' for ns in odd_ns]

fig, ax = plt.subplots(figsize=(10, 4))
bars = ax.bar([f'e{ns}' for ns in odd_ns], actual, color=colors, edgecolor='black', linewidth=0.5)
ax.axhline(dirichlet, color='red', linestyle='--', linewidth=1.2, label=f'Dirichlet asymptote {dirichlet:.1f}%')
ax.set_xlabel('N-shape (p mod 16, odd residues only)')
ax.set_ylabel('% of primes ≤ 1000')
ax.set_title('Prime leaf density by N-shape — deviations from Dirichlet are L-function oscillations')
gap_patch = mpatches.Patch(color='silver', label='Monster gap {e₁, e₁₁, e₁₅} — Telperion silver')
ax.legend(handles=[gap_patch, ax.get_lines()[0]])
plt.tight_layout()
plt.savefig('01_nshape_distribution.png', dpi=150)
plt.show()

## Prime gap fractal

In [ ]:
gf = prime_gap_fractal(primes)

print(f'Number of gaps:       {gf["n_gaps"]}')
print(f'Mean gap:             {gf["mean_gap"]:.4f}')
print(f'Min / Max gap:        {gf["min_gap"]} / {gf["max_gap"]}')
print(f'Self-similar ratio:   {gf["self_similar_ratio"]:.6f}')
print()
print('Gap stats by N-shape (p mod 16 of smaller prime):')
print(f'  {"N-shape":>8}  {"Count":>6}  {"Mean gap":>9}  {"Max gap":>8}')
for ns, st in sorted(gf['nshape_gap_stats'].items()):
    gap_flag = ' ← SILVER' if ns in NIEMEIER_GAP else ''
    print(f'  e{ns:2d}      {st["count"]:6d}  {st["mean"]:9.3f}  {st["max"]:8d}{gap_flag}')

In [ ]:
fig, axes = plt.subplots(1, 2, figsize=(14, 4))

# Left: gaps vs. prime index
axes[0].plot(gf['gaps'], color='#334466', linewidth=0.4, alpha=0.8)
axes[0].plot(gf['running_mean'], color='red', linewidth=1.5, label='Running mean')
axes[0].set_xlabel('Prime index')
axes[0].set_ylabel('Gap to next prime')
axes[0].set_title('Prime gaps — fractal envelope (oscillations = Riemann zeros)')
axes[0].legend()

# Right: gap distribution histogram (log-log shows fractal scaling)
gaps_arr = np.array(gf['gaps'])
unique_gaps, gap_counts = np.unique(gaps_arr, return_counts=True)
axes[1].loglog(unique_gaps, gap_counts, 'o', markersize=4, color='#334466')
axes[1].set_xlabel('Gap size (log scale)')
axes[1].set_ylabel('Frequency (log scale)')
axes[1].set_title('Gap frequency — power law tail signals fractal character')

plt.tight_layout()
plt.savefig('01_prime_gap_fractal.png', dpi=150)
plt.show()

## Monster gap leaf density

The three silver N-shapes {e₁, e₁₁, e₁₅} should each hold ~12.5% of primes  
asymptotically (Dirichlet).  The Fermat Monster Engine (v0.300) proves they  
are unreachable by any Niemeier A/D/E root system.  Only Monster primes  
{p=17, 11, 59, 31, 47} activate them.

The leaves with these N-shapes are the 'most un-extinctable' — they lie in  
the algebraic positions that cannot even be DESCRIBED by the regular structure  
that the Niemeier lattices provide.

In [ ]:
gap_primes = [p for p in primes if p % 16 in NIEMEIER_GAP]
print(f'Monster gap primes ≤ {N}: {len(gap_primes)}')
print(f'First 30: {gap_primes[:30]}')
print()
for ns in sorted(NIEMEIER_GAP):
    ps = [p for p in primes if p % 16 == ns]
    moonshine = [p for p in ps if p in MOONSHINE_PRIMES]
    print(f'  e{ns}: {len(ps)} primes, moonshine in set: {moonshine[:5]}')